# Phase 12 — Multi-Frame Extraction (8 evenly-spaced frames per video)

**Why:** EF is fundamentally a motion measurement (how much the left ventricle contracts between diastole and systole). The single-middle-frame model (used in notebooks 02-11) cannot see this motion, which is the main reason its accuracy (MAE~7.3) trails published video-based methods (MAE~4-6). This notebook extracts 8 evenly-spaced frames per video -- enough to capture the general contraction pattern without the storage/compute cost of full video (typically 50-200+ frames).

**Storage warning (learned the hard way earlier in this project):** 8 frames/video x 10,030 videos x ~37KB/frame (112x112x3 uint8) is approximately **3GB** -- on top of whatever is already used on the free 15GB Drive. **Run the storage-check cell first** and follow its guidance before extracting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob
import numpy as np
import pandas as pd

PROJECT_ROOT = '/content/drive/MyDrive/echonet_project'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data/raw', 'EchoNet-Dynamic', 'EchoNet-Dynamic')
VIDEOS_DIR = os.path.join(DATA_DIR, 'Videos')
FULL_CSV = os.path.join(DATA_DIR, 'FileList.csv')
MULTIFRAME_CACHE_DIR = os.path.join(PROJECT_ROOT, 'data/multiframe', 'frames')
os.makedirs(MULTIFRAME_CACHE_DIR, exist_ok=True)

df = pd.read_csv(FULL_CSV)
filename_col = 'FileName' if 'FileName' in df.columns else [c for c in df.columns if 'file' in c.lower() and 'name' in c.lower()][0]
print('Total videos:', len(df))

## Storage check (do this before extracting)

In [ ]:
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
free_gb = free / (1024**3)
N_FRAMES = 8
estimated_gb = len(df) * N_FRAMES * 37_000 / (1024**3)
print(f'Free space on Drive: {free_gb:.2f} GB')
print(f'Estimated space needed for {N_FRAMES}-frame extraction: ~{estimated_gb:.2f} GB')

if free_gb < estimated_gb + 1.0:  # 1GB safety margin
    print('\nWARNING: may not have enough free space. Consider freeing space first, e.g.:')
    old_single_frame_dir = os.path.join(PROJECT_ROOT, 'data/full', 'frames')
    if os.path.isdir(old_single_frame_dir):
        n_old = len(glob.glob(os.path.join(old_single_frame_dir, '*.npy')))
        print(f"  - The old single-frame cache at data/full/frames has {n_old} files; "
              f"you likely no longer need it once multi-frame extraction succeeds.")
    print('  - Check checkpoints/ for old experiment folders (e.g. baseline_resnet18_subsample,')
    print('    mc_dropout_subsample_v1) that are superseded and safe to delete.')
else:
    print('\nLooks like enough space is available. Proceeding is reasonable.')

## Resumable multi-frame extraction

Each video's 8 frames are saved as a single stacked array `(8, H, W, 3)` per video (one .npy file), so partial per-video work is not left half-done. Already-completed videos are skipped, so this is safe to stop and resume across sessions.

In [ ]:
!pip install --quiet opencv-python-headless
import cv2, time

def get_evenly_spaced_frames(video_path, n_frames=8):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return None
    indices = np.linspace(0, max(total_frames - 1, 0), n_frames).astype(int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok:
            cap.release()
            return None
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return np.stack(frames, axis=0)  # (n_frames, H, W, 3)

start_time = time.time()
cached, missing, skipped = 0, 0, 0
for i, row in df.iterrows():
    fname = str(row[filename_col])
    if not fname.endswith('.avi'):
        fname = fname + '.avi'
    cache_path = os.path.join(MULTIFRAME_CACHE_DIR, fname.replace('.avi', '.npy'))
    if os.path.exists(cache_path):
        skipped += 1
        continue
    video_path = os.path.join(VIDEOS_DIR, fname)
    if not os.path.exists(video_path):
        missing += 1
        continue
    frames = get_evenly_spaced_frames(video_path, n_frames=N_FRAMES)
    if frames is not None:
        np.save(cache_path, frames)
        cached += 1
    else:
        missing += 1
    if (cached + skipped) % 300 == 0:
        elapsed_min = (time.time() - start_time) / 60
        print(f'[{i+1}/{len(df)}] newly cached={cached}, already done={skipped}, missing={missing}, elapsed={elapsed_min:.1f} min')

print(f'DONE this run. newly cached={cached}, already done={skipped}, missing={missing}')
print('If this session gets disconnected partway, just re-run this cell -- it skips completed videos.')

## Verify

Quick sanity check: load a couple of cached clips and confirm the shape is as expected.

In [ ]:
sample_files = glob.glob(os.path.join(MULTIFRAME_CACHE_DIR, '*.npy'))[:3]
for f in sample_files:
    arr = np.load(f)
    print(os.path.basename(f), arr.shape, arr.dtype)

n_total_cached = len(glob.glob(os.path.join(MULTIFRAME_CACHE_DIR, '*.npy')))
print(f'\nTotal videos with cached multi-frame clips: {n_total_cached} / {len(df)}')
if n_total_cached < len(df) * 0.98:
    print('Extraction is not yet complete -- re-run the extraction cell above to continue.')
else:
    print('Extraction looks complete. Proceed to notebook 13 for multi-frame model training.')